
# 2D slice shape descriptors

This notebook computes **per-object 2D shape descriptors** from 3D membrane label volumes:

1. Align and filter input annotations (spots ↔ nuclei ↔ membranes).
2. For each membrane label at a timepoint, extract a **2D crop** at the object's centroid Z-slice.
3. Compute classical shape descriptors (area, perimeter, eccentricity, solidity, extent, major/minor axis lengths, and max Feret diameter) for each 2D crop.
4. Unpack the features into flat columns and save a tidy CSV.


## 1) Imports & Configuration

In [ ]:

import os
import numpy as np
import pandas as pd
import tifffile
from skimage.measure import regionprops, label
from skimage import measure
from tqdm import tqdm

# Input tables
MEM_NUC_CSV = r'D:/Mari_Sixth_Dataset_Analysis/mem_nuc_overlap_table.csv' #calculated in "assign membrane objects to nuclei.ipynb"
SPOTS_CSV   = r'D:/Mari_Sixth_Dataset_Analysis/full_df_for_classification.csv' #calculated in 

# Source label stacks (3D TIFF per timepoint); {t} is 1-based index in filenames
LABEL_TIF_TMPL = (
    r'D:/Mari_Sixth_Dataset_Analysis/split_nuclei_membrane_raw/split_cellpose_results_relabelled/'
    r'final_relabelled_timelapse_sixth_dataset-{t}.tif'
)

# Output CSV
OUT_CSV = r'D:/Mari_Sixth_Dataset_Analysis/nuclei_membrane_tracking/membrane_manual_dataset_with2D_new.csv'

# Parameters
TOTAL_TIMEPOINTS = 360
BBOX_HALF_SIZE   = 75
PAD_WITH         = 0



## 2) Data Loading & Preprocessing
We merge **spots** with the nuc↔mem mapping so each row carries the membrane label for the nucleus at that timepoint.
Duplicates are removed to enforce uniqueness for `(Spot frame, nuc_label)` and `(Spot frame, mem_label)` before feature extraction.


In [ ]:

mem_nuc_df = pd.read_csv(MEM_NUC_CSV)
for c in ('nuc_label','mem_label','t'):
    if c in mem_nuc_df.columns:
        mem_nuc_df[c] = pd.to_numeric(mem_nuc_df[c], errors='coerce').astype('Int64')

spots_df = pd.read_csv(SPOTS_CSV)
if 'Spot frame' not in spots_df.columns:
    raise KeyError("SPOTS_CSV must contain column 'Spot frame'.")

if 'mem_label' not in spots_df.columns:
    merged_df_full = spots_df.merge(
        mem_nuc_df[['nuc_label','mem_label','t']],
        left_on=['nuc_label','Spot frame'],
        right_on=['nuc_label','t'],
        how='left'
    )
else:
    merged_df_full = spots_df.copy()

merged_df_full = merged_df_full.loc[
    ~merged_df_full.duplicated(subset=['Spot frame','nuc_label'], keep=False)
]
merged_df_full_cropped = merged_df_full.loc[
    ~merged_df_full.duplicated(subset=['Spot frame','mem_label'], keep=False)
]

merged_df_full_cropped = merged_df_full_cropped.copy()
merged_df_full_cropped['features_array'] = [[] for _ in range(len(merged_df_full_cropped))]

print(f"rows after cleaning: {len(merged_df_full_cropped)}")



## 3) Helper Functions
Small utilities for: centroid mapping, binary crop extraction, and 2D descriptors.


In [ ]:

# Map label to centroid (z, y, x)
def measure_objects(segmented_image: np.ndarray) -> dict:
    return {prop.label: prop.centroid for prop in regionprops(segmented_image)}

# Extract 2D binary crops at centroid Z for given labels
def extract_binary_crops(segmented_image: np.ndarray, membrane_labels: list, bbox_half: int, label_centroid_map: dict) -> np.ndarray:
    crops = []
    H = 2 * bbox_half
    for lab in membrane_labels:
        if pd.isna(lab):
            crops.append(np.zeros((H, H), dtype=np.uint8))
            continue
        lab = int(lab)
        cy, cx = int(round(label_centroid_map[lab][1])), int(round(label_centroid_map[lab][2]))
        cz = int(round(label_centroid_map[lab][0]))
        r0, r1 = cy - bbox_half, cy + bbox_half
        c0, c1 = cx - bbox_half, cx + bbox_half
        sl = segmented_image[cz, r0:r1, c0:c1]
        sl = np.where(sl == lab, 1, 0).astype(np.uint8)
        if sl.shape != (H, H):
            pad_r = H - sl.shape[0]
            pad_c = H - sl.shape[1]
            sl = np.pad(sl, ((0,pad_r),(0,pad_c)), mode='constant', constant_values=0)
        crops.append(sl)
    return np.stack(crops, axis=0)

# Compute 2D shape descriptors for a binary slice

def shape_descriptors_2d(binary_image: np.ndarray) -> list:
    labeled = label(binary_image)
    regions = regionprops(labeled)
    if not regions:
        return [np.nan]*8
    r = max(regions, key=lambda rr: rr.area)
    return [
        r.area, r.perimeter, r.eccentricity, r.solidity, r.extent,
        getattr(r, 'axis_major_length', np.nan), getattr(r, 'axis_minor_length', np.nan),
        getattr(r, 'feret_diameter_max', np.nan),
    ]

# Vectorized wrapper for a stack of crops

def descriptors_for_stack(binary_stack: np.ndarray, names: list) -> np.ndarray:
    out = []
    for i in range(binary_stack.shape[0]):
        out.append(shape_descriptors_2d(binary_stack[i]))
    return np.asarray(out)



## 4) Feature Extraction Loop
For each timepoint `t` (1-based), read the label stack, pad it, subset rows for `Spot frame == t-1`,
build centroid map, extract 2D crops, compute descriptors, and attach them to rows.


In [ ]:

feature_names = [
    'mem_2d_area','mem_2d_perimeter','mem_2d_eccentricity','mem_2d_solidity',
    'mem_2d_extent','mem_2d_axis_major_length','mem_2d_axis_minor_length','mem_2d_feret_diameter_max'
]

out_rows = []
for t in tqdm(range(0, TOTAL_TIMEPOINTS), desc='timepoints'):
    seg = tifffile.imread(LABEL_TIF_TMPL.format(t=t))
    seg_padded = np.pad(seg, ((0,0),(BBOX_HALF_SIZE,BBOX_HALF_SIZE),(BBOX_HALF_SIZE,BBOX_HALF_SIZE)),
                         mode='constant', constant_values=PAD_WITH)

    sub = merged_df_full_cropped.loc[merged_df_full_cropped['Spot frame'] == (t)].copy()
    if sub.empty:
        continue
    mem_labels = sub['mem_label'].to_list()

    centroid_map = measure_objects(seg_padded)
    crops = extract_binary_crops(seg_padded, mem_labels, BBOX_HALF_SIZE, centroid_map)

    feats = descriptors_for_stack(crops, feature_names)
    feat_df = pd.DataFrame(feats, columns=feature_names, index=sub.index)

    sub = pd.concat([sub.drop(columns=['features_array'], errors='ignore'), feat_df], axis=1)
    out_rows.append(sub)

mem_2d_df = pd.concat(out_rows, ignore_index=True) if out_rows else pd.DataFrame(columns=list(merged_df_full_cropped.columns)+feature_names)
print(f'final rows: {len(mem_2d_df)}')


## 5) Save

In [ ]:

os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
mem_2d_df.to_csv(OUT_CSV, index=False)
OUT_CSV



---
### Notes & Tips
- Increase `BBOX_HALF_SIZE` if objects are larger.
- Descriptors come from `skimage.measure.regionprops` on a single slice (largest component if fragmented).
